In [0]:
# Create widgets for source_system, table_name, and landing_timestamp for passing dynamic data
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")



In [0]:
landing_path = f"/Volumes/adb_hospital/staging/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check if the landing path exists
try:
    dbutils.fs.ls(landing_path)
    path_exists = True
except Exception:
    path_exists = False

if not path_exists:
    dbutils.notebook.exit(f"FAILED: Landing path not found -> {landing_path}")

if source_file_format == "csv":
    landing_df = (spark.read.format("csv")
                 .option("header", True)
                 .option("inferSchema", True)
                 .load(landing_path))
else:
    landing_df = spark.read.format("parquet").load(landing_path)


In [0]:
import pyspark.sql.functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_hospital.bronze.{table_name}")